What to do:
1. Define the best temporary location for each store => DONE
2. Take the code bellow and replace the randomisation by the exel files (only 2 Weeks). => DONE
3. Run the code and get the loading results displayed in a graph => DONE
4. Loop the Supermaket locations (4 loops) to find the best location for each of them by calculating the average minimum load of the average of all the lines. => ToDo
5. Step 2 Done!

In [56]:
import copy
import pandapower as pp
from pandapower.plotting.plotly import simple_plotly
from pandapower.plotting.plotly import pf_res_plotly


net = pp.networks.create_cigre_network_mv(with_der=False)


net.load['p_mw']= net.load['p_mw'] / 2
net.load['q_mvar']= net.load['q_mvar'] / 2

pp.runpp(net)
print(net.load)

# print(net.res_bus)
# print(net.res_line.loading_percent)
# print(net.res_trafo.loading_percent)

# simple_plotly(net)
# pf_res_plotly(net)

         name  bus      p_mw    q_mvar  const_z_percent  const_i_percent  \
0     Load R1    1  7.497000  1.522331              0.0              0.0   
1     Load R3    3  0.138225  0.034642              0.0              0.0   
2     Load R4    4  0.215825  0.054091              0.0              0.0   
3     Load R5    5  0.363750  0.091164              0.0              0.0   
4     Load R6    6  0.274025  0.068677              0.0              0.0   
5     Load R8    8  0.293425  0.073539              0.0              0.0   
6    Load R10   10  0.237650  0.059561              0.0              0.0   
7    Load R11   11  0.164900  0.041328              0.0              0.0   
8    Load R12   12  7.497000  1.522331              0.0              0.0   
9    Load R14   14  0.104275  0.026134              0.0              0.0   
10   Load CI1    1  2.422500  0.796237              0.0              0.0   
11   Load CI3    3  0.112625  0.069799              0.0              0.0   
12   Load CI

1. Best temporary locations for each store:
ICA=> Average 1211 kW/h
Mathem => Average 787 kW/h
Postnord => Average 1000 kW/h
Airmee=> Average 317 kW/h

With the Graph Visualisation, the best locations for new loads seems to be Bus 6,8,14,4 for the best repartition of power load
Let's define 
ICA => Bus 8
Mathem => Bus 6
Postnord => 14
Airmee => 4

In [57]:
import sys
import pandapower as pp
import pandapower.networks as nw
import pandas as pd
import nbformat
print(nbformat.__version__)
import matplotlib.pyplot as plt
from pandapower.plotting.plotly import simple_plotly
from pandapower.plotting.plotly import pf_res_plotly

# ------------- 1. create network and apply initial change -------------
net = nw.create_cigre_network_mv(with_der=False)

# scale down loads
net.load['p_mw'] = net.load['p_mw'] / 2.0
net.load['q_mvar'] = net.load['q_mvar'] / 2.0

# ------------- 2. Read the Excel File -------------
file_path = "Load_profiles_Freight transports.xlsx"
load_profile = pd.read_excel(file_path)

load_profile["Datetime"] = pd.to_datetime(load_profile["Date"].astype(str) + " " + load_profile["Time"].astype(str))
load_profile = load_profile.set_index("Datetime")

# Only keep the 4 shop columns
shops = ["ICA", "Mathem", "Postnord", "Airmee"]
load_profile = load_profile[shops]

# ------------- 3. Mapping shops to buses -------------

# Here make 4 loops to find the best location for each shop
shop_to_bus = {
    "ICA": 0,
    "Mathem": 0,
    "Postnord": 0,
    "Airmee": 0,
}

best_case_mean_loads = sys.float_info.max # initialize with a large number
best_shop_to_bus = None
best_res_lines = None
best_net = None

list_of_mean_loads = []

#test version


"""
for ICA_index in load_profile.index:
    for Mathem_index in load_profile.index:
        for Postnord_index in load_profile.index:
            for Airmee_index in load_profile.index:
                
"""

for ICA_index in [12]:
    for Mathem_index in [12]:
        for Postnord_index in [12]:
            for Airmee_index in [12]:

                shop_to_bus = {
                    "ICA": ICA_index,
                    "Mathem": Mathem_index,
                    "Postnord": Postnord_index,
                    "Airmee": Airmee_index,
                }


                # ------------- 4. results DataFrame for line loading -------------
                line_indices = list(net.line.index)
                col_names = [f"line_{i}" for i in line_indices]
                res_lines = pd.DataFrame(index=load_profile.index, columns=col_names, dtype=float)


                # ------------- 5. simulation loop -------------
                for ts in load_profile.index:

                    # assign each shop load to its bus
                    for shop, bus in shop_to_bus.items():
                        # locate the load element connected to that bus
                        idx = net.load[net.load.bus == bus].index
                        if not idx.empty:
                            
                            net.load.at[idx[0], 'p_mw'] += load_profile.at[ts, shop] / 1000.0  # Change unit kW to MW
                            
                    # run power flow
                    try:
                        pp.runpp(net, calculate_voltage_angles=False)
                        loading = net.res_line['loading_percent']
                        for li in line_indices:
                            res_lines.at[ts, f"line_{li}"] = loading.at[li]

                        
                    except Exception as e:
                        print(f"Power flow failed at {ts}: {e}")
                        res_lines.loc[ts, :] = np.nan

                avg_loads = res_lines.max().mean()
                list_of_mean_loads.append(avg_loads)
                if avg_loads < best_case_mean_loads:
                    best_case_mean_loads = avg_loads
                    best_shop_to_bus = shop_to_bus.copy()
                    best_res_lines = res_lines.copy()  


                    best_net = copy.deepcopy(net)
                
                
                
                    

                    

# ------------- 6. summary Average Load Stats -------------
print("Per-line max loading (%) over 2 weeks:")
print(best_res_lines.max().sort_values(ascending=False))

# ------------- 7. plot the Load of each line -------------
plt.figure(figsize=(14, 6))

for col in best_res_lines.columns:
    plt.plot(best_res_lines.index, best_res_lines[col], linewidth=0.9, label=col)

plt.xlabel("Time")
plt.ylabel("Line loading (%)")
plt.title("Line loading percent — 2 weeks (hourly)")
plt.grid(alpha=0.3)

# place legend outside the plot for readability
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), ncol=1, fontsize=8)

plt.tight_layout()
plt.show()

simple_plotly(best_net)
pf_res_plotly(best_net)



# ------------- 10. save results -------------
# res_lines.to_csv("line_loading_2weeks_hourly.csv")
# print("Saved CSV: line_loading_2weeks_hourly.csv")


5.10.4


Power flow failed at 2026-01-02 05:00:00+01:00: Power Flow nr did not converge after 10 iterations!


NameError: name 'np' is not defined

In [ ]:
print("Best shop to bus mapping:", best_shop_to_bus)
print("List of mean:"  ,list_of_mean_loads)
print(len(list_of_mean_loads))
print(load_profile.head())

print(res_lines)

Best shop to bus mapping: {'ICA': 12, 'Mathem': 12, 'Postnord': 12, 'Airmee': 12}
List of mean: [12.461011244236717]
1
                                  ICA      Mathem    Postnord      Airmee
Datetime                                                                 
2026-01-01 00:00:00+01:00  853.470265  558.823304  476.323357  179.548523
2026-01-01 01:00:00+01:00  862.974883  521.901614  473.711723  161.847819
2026-01-01 02:00:00+01:00  839.105117  549.134102  450.264885  174.293495
2026-01-01 03:00:00+01:00  817.136709  521.987351  458.649281  149.116478
2026-01-01 04:00:00+01:00  838.921546  516.426191  418.629654  148.731466
                              line_0     line_1     line_2     line_3  \
Datetime                                                                
2026-01-01 00:00:00+01:00  44.719335  45.130391  17.577095  13.133517   
2026-01-01 01:00:00+01:00  44.719335  45.130391  17.577095  13.133517   
2026-01-01 02:00:00+01:00  44.719335  45.130391  17.577095  13.133517  